# 022-adaptive-gate analysis

# Imports

In [ ]:
import json
import os
from dataclasses import replace
from itertools import product

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.drone_frames import load_frames
from uavloc.data.multidate import load_date
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.matching.benchmark import benchmark_draws, error_stats, paired_frame_bootstrap, score_draws
from uavloc.matching.correlator import PositionCorrelator
from uavloc.matching.tuning import gate_key
from uavloc.track import REPO_ROOT, experiment_dir

# Params

In [ ]:
ARMS = {"007.1": "ungated", "022.1": "static full", "022.2": "static accept-reject", "022.3": "risk tau 0.25", "022.4": "risk multi-tau"}
PRIMARY_NOISE_M = 10.0
NOISE_M = [5.0, 10.0, 15.0]
BAR_HARM_SHARE = 0.042
BAR_APPLIED_SHARE = 0.60
BAR_MEAN_COST_M = 0.5
N_BOOTSTRAP = 10_000
CI_PCT = 95
SEED = 42
PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
N_DIRECTIONS = 8
DEVICE = "mps"
FRAMES_DATASET = "village-crossdate-frames"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
MATCH_GSD_M = 0.385
MAP_DATE = "newest"
TEMPERATURE_GRID = [0.25, 0.5, 1.0, 2.0]
S_MAX_GRID_M = [2.0, 3.0, 5.0]
C_MIN_GRID = [0.01, 0.03, 0.05]
MIN_APPLIED_SHARE = 0.5
HARM_MARGIN_M = 2.0
RUN_CONFIG = {"method": "segmentation", "matching": "weighted-ncc", "class_weights": (0.0, 1.0, 1.0, 0.5, 0.0, 0.5),
              "window": "rect", "search_scale": 1.7, "search_radius_m": None, "prior_sigma_m": None, "gate": "none",
              "fov_w_deg": 60.0, "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0}

# Data

In [ ]:
exp_root = experiment_dir("022-adaptive-gate")
records = {"007.1": pd.read_csv(experiment_dir("007-search-prior-and-gate") / "artifacts/records-007.1.csv")}
records.update({run: pd.read_csv(exp_root / f"artifacts/records-{run}.csv") for run in ARMS if run.startswith("022")})
{run: len(r) for run, r in records.items()}

In [ ]:
for r in records.values():
    r["applied"] = r["decision"].isin(["accept", "across"]) if "decision" in r else True
    r["harm"] = r["error_m"] > r["prior_error_m"] + HARM_MARGIN_M
records["007.1"]["applied"] = records["007.1"]["status"] == "ok"

# Arms

In [ ]:
def arm_row(run: str, noise_m: float) -> dict[str, float]:
    r = records[run][records[run]["noise_m"] == noise_m]
    return {"arm": ARMS[run], "run": run, "noise_m": noise_m, "mean_m": r["error_m"].mean(), "median_m": r["error_m"].median(),
            "p90_m": r["error_m"].quantile(0.9), "max_m": r["error_m"].max(), "harm": r["harm"].mean(),
            "applied": r["applied"].mean()}

In [ ]:
table = pd.DataFrame([arm_row(run, n) for run in ARMS for n in NOISE_M])
table[table["noise_m"] == PRIMARY_NOISE_M].round(3)

In [ ]:
table.pivot(index="arm", columns="noise_m", values="mean_m").round(2)

# Bar

In [ ]:
def bar_check(run: str) -> pd.DataFrame:
    row = arm_row(run, PRIMARY_NOISE_M)
    ungated = arm_row("007.1", PRIMARY_NOISE_M)
    return pd.DataFrame({"value": [row["harm"], row["applied"], row["mean_m"]],
                         "bar": [BAR_HARM_SHARE, BAR_APPLIED_SHARE, ungated["mean_m"] + BAR_MEAN_COST_M],
                         "met": [row["harm"] <= BAR_HARM_SHARE, row["applied"] >= BAR_APPLIED_SHARE,
                                 row["mean_m"] <= ungated["mean_m"] + BAR_MEAN_COST_M]},
                        index=["harm", "applied", "mean_m"]).round(3)

In [ ]:
{run: bar_check(run) for run in ("022.2", "022.3", "022.4")}

# Paired

In [ ]:
def paired(run: str, reference: str) -> dict[str, float]:
    return {"run": run, "reference": reference,
            **paired_frame_bootstrap(records[run], records[reference], PRIMARY_NOISE_M, N_BOOTSTRAP, CI_PCT, SEED)}

In [ ]:
pd.DataFrame([paired("022.4", "022.2"), paired("022.3", "022.4"), paired("022.1", "022.2"), paired("022.4", "007.1"),
              paired("022.2", "007.1"), paired("022.4", "022.1")]).round(3)

# Places

In [ ]:
per_place = pd.concat([records[run][records[run]["noise_m"] == PRIMARY_NOISE_M].assign(run=run) for run in ARMS])
per_place["place"] = per_place["frame"].str.split("-").str[0]
per_place.groupby(["place", "run"]).agg(mean_m=("error_m", "mean"), harm=("harm", "mean"), applied=("applied", "mean")).round(3)

# Harm

In [ ]:
def accepted_harm(run: str) -> dict[str, float]:
    r = records[run][(records[run]["noise_m"] == PRIMARY_NOISE_M) & records[run]["applied"]]
    bad = r[r["harm"]]
    return {"run": run, "accepted": len(r), "harmful": len(bad), "harmful_share": len(bad) / max(len(r), 1),
            "mean_error_harmful_m": bad["error_m"].mean(), "max_error_m": bad["error_m"].max()}

In [ ]:
pd.DataFrame([accepted_harm(run) for run in ("022.1", "022.2", "022.3", "022.4")]).round(3)

# Weights

In [ ]:
runs = [json.loads(line) for line in (exp_root / "runs.jsonl").read_text().splitlines()]
weights = {r["run"]: r["config"]["risk_weights"] for r in runs if r["config"].get("risk_weights")}
pd.concat({run: pd.DataFrame(w) for run, w in weights.items()}, axis=1).round(2)

In [ ]:
{r["run"]: {k: r["config"][k] for k in ("p_max", "temperature", "gate_s_max_m", "gate_c_min")} for r in runs}

# Optimum

In [ ]:
frames = {p: [f for f in load_frames(FRAMES_DATASET)
              if json.loads((f.image_path.parent / "meta.json").read_text())["place"] == p] for p in PLACES}
maps = {p: load_date(p, MAP_DATE) for p in PLACES}
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, "lraspp")

In [ ]:
base_config = correlator_config(RUN_CONFIG)
correlators = {p: PositionCorrelator(SegmentationFeatures(model, model, maps[p], "all", MATCH_GSD_M, MATCH_GSD_M, DEVICE),
                                     base_config) for p in PLACES}

In [ ]:
draws = {p: [d for n in NOISE_M for d in benchmark_draws(correlators[p], frames[p], n, N_DIRECTIONS, SEED)] for p in PLACES}

In [ ]:
def best_values(place: str, noise_m: float) -> dict[str, float]:
    """Best static gate (the 007 grid, `full`) for one place at one injected level, and what it scores."""
    level = [d for d in draws[place] if d.noise_m == noise_m]
    rows = []
    for t, s_max, c_min in product(TEMPERATURE_GRID, S_MAX_GRID_M, C_MIN_GRID):
        stats = error_stats(score_draws(level, replace(base_config, temperature=t, gate="full", gate_s_max_m=s_max,
                                                       gate_c_min=c_min)))
        rows.append({"temperature": t, "s_max_m": s_max, "c_min": c_min, "harm": stats["harm_share"],
                     "mean_m": stats["mean_position_error_m"], "applied": stats["accept_share"] + stats["across_share"]})
    best = min(rows, key=lambda r: gate_key({"accept_share": r["applied"], "across_share": 0.0, "harm_share": r["harm"],
                                             "mean_position_error_m": r["mean_m"]}, MIN_APPLIED_SHARE))
    return {"place": place.split("-")[0], "noise_m": noise_m, **best}

In [ ]:
optimum = pd.DataFrame([best_values(p, n) for p in PLACES for n in NOISE_M])
optimum.round(3)

# Plots

In [ ]:
fig_arms, ax = plt.subplots(figsize=(6, 4))
for run, label in ARMS.items():
    r = table[table["run"] == run]
    ax.plot(r["harm"], r["mean_m"], "o-", label=label)
ax.set_xlabel("harm share"); ax.set_ylabel("mean error, m")
ax.set_title("mean error vs harm, points are 5 / 10 / 15 m"); ax.legend(); ax.grid(alpha=.3)
fig_arms.tight_layout()
(exp_root / "figures").mkdir(exist_ok=True)
fig_arms.savefig(exp_root / "figures/mean-vs-harm.png", dpi=150)

# Save

In [ ]:
table.to_csv(exp_root / "artifacts/arm-table.csv", index=False)
optimum.to_csv(exp_root / "artifacts/optimum-by-place-and-level.csv", index=False)